In [7]:
from pathlib import Path
import re
from collections import Counter

import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

from rdkit import Chem
from rdkit.Chem import Draw


inputFile = Path("Ebola_High_pPotency_molecule_pathway1/Ebola_High_pPotency_molecule_pathway1_ranked_pathways.txt")
outputDir = Path("pathway_figures")
outputDir.mkdir(exist_ok=True)



In [8]:



# =========================
# Parsing
# =========================
def parsePathwayBlocks(filePath):
    text = Path(filePath).read_text().strip()
    rawBlocks = re.split(r"\n(?=ranking\s+\d+\n)", text)

    pathwayList = []

    for block in rawBlocks:
        block = block.strip()
        if not block.startswith("ranking"):
            continue

        lines = [line.strip() for line in block.splitlines() if line.strip()]

        rank = None
        finalScore = None
        atomicEconomy = None
        reactionList = []
        ruleList = []
        enthalpyList = []
        inReactionSection = False

        for line in lines:
            if line.startswith("ranking"):
                matchObj = re.search(r"ranking\s+(\d+)", line)
                if matchObj:
                    rank = int(matchObj.group(1))

            elif line.startswith("final score"):
                finalScore = line.replace("final score", "").strip()

            elif line.startswith("atomic economy"):
                atomicEconomy = line.replace("atomic economy", "").strip()

            elif line.startswith("reaction SMILES"):
                inReactionSection = True

            elif inReactionSection:
                if ">>" in line:
                    reactionList.append(line)
                elif re.match(r"^rule\d+_\d+$", line):
                    ruleList.append(line)
                else:
                    enthalpyList.append(line)

        pathwayList.append(
            {
                "rank": rank,
                "finalScore": finalScore,
                "atomicEconomy": atomicEconomy,
                "reactionList": reactionList,
                "ruleList": ruleList,
                "enthalpyList": enthalpyList,
            }
        )

    return pathwayList


def splitReactionSmiles(reactionSmiles):
    reactantSide, productSide = reactionSmiles.split(">>")
    reactantList = [item.strip() for item in reactantSide.split(".") if item.strip()]
    productList = [item.strip() for item in productSide.split(".") if item.strip()]
    return reactantList, productList


# =========================
# Chemistry helpers
# =========================
def canonicalizeSmiles(smiles):
    molObj = Chem.MolFromSmiles(smiles)
    if molObj is None:
        return smiles
    return Chem.MolToSmiles(molObj)


def canonicalizeList(smilesList):
    return [canonicalizeSmiles(smiles) for smiles in smilesList]


def multisetIntersection(listA, listB):
    counterA = Counter(listA)
    counterB = Counter(listB)
    shared = []
    for key in counterA:
        n = min(counterA[key], counterB.get(key, 0))
        shared.extend([key] * n)
    return shared


def removeOneListFromAnother(fullList, toRemove):
    counterRemove = Counter(toRemove)
    result = []
    for item in fullList:
        if counterRemove[item] > 0:
            counterRemove[item] -= 1
        else:
            result.append(item)
    return result


# =========================
# RDKit drawing helpers
# =========================
def smilesToPil(smiles, size=(260, 180)):
    molObj = Chem.MolFromSmiles(smiles)
    if molObj is None:
        return None
    return Draw.MolToImage(molObj, size=size)


def addMolImage(ax, pilImg, xy, zoom=0.40):
    if pilImg is None:
        ax.text(
            xy[0], xy[1], "Invalid\nSMILES",
            ha="center", va="center", fontsize=8,
            bbox=dict(boxstyle="round", facecolor="white", edgecolor="black")
        )
        return

    imageBox = OffsetImage(pilImg, zoom=zoom)
    annotationBox = AnnotationBbox(imageBox, xy, frameon=False)
    ax.add_artist(annotationBox)


def drawMolSet(ax, smilesList, xStart, yCenter, xStep=2.4, plusFontSize=18, zoom=0.40):
    xPositions = []
    currentX = xStart

    for smiles in smilesList:
        xPositions.append(currentX)
        pilImg = smilesToPil(smiles)
        addMolImage(ax, pilImg, (currentX, yCenter), zoom=zoom)
        currentX += xStep

    for i in range(len(xPositions) - 1):
        xMid = (xPositions[i] + xPositions[i + 1]) / 2
        ax.text(xMid, yCenter, "+", ha="center", va="center", fontsize=plusFontSize)

    if len(xPositions) == 0:
        return xStart

    return xPositions[-1]


# =========================
# Pathway compression
# =========================
def buildChainedPathway(pathwayDict):
    """
    Convert reaction steps into a continuous chain:
    mainChain[0] --step1--> mainChain[1] --step2--> mainChain[2] ...

    Extra reactants/products that are not part of the propagated intermediate
    are stored as side species for each step.
    """
    reactionList = pathwayDict["reactionList"]
    parsedSteps = []

    for reactionSmiles in reactionList:
        reactantList, productList = splitReactionSmiles(reactionSmiles)
        reactantList = canonicalizeList(reactantList)
        productList = canonicalizeList(productList)
        parsedSteps.append({"reactants": reactantList, "products": productList})

    if len(parsedSteps) == 0:
        return {"mainChain": [], "stepInfo": []}

    # Infer carried intermediates using overlap between consecutive steps
    carriedProducts = []
    for i in range(len(parsedSteps) - 1):
        currentProducts = parsedSteps[i]["products"]
        nextReactants = parsedSteps[i + 1]["reactants"]
        shared = multisetIntersection(currentProducts, nextReactants)
        carriedProducts.append(shared)

    # Build main chain
    mainChain = []
    stepInfo = []

    # Initial displayed starting set = reactants of first step excluding
    # anything that is actually a carried-over reagent from a previous step
    firstMainInput = parsedSteps[0]["reactants"]
    mainChain.append(firstMainInput)

    for i, step in enumerate(parsedSteps):
        reactants = step["reactants"]
        products = step["products"]

        if i == 0:
            incomingMain = reactants
        else:
            incomingMain = carriedProducts[i - 1]

        outgoingMain = carriedProducts[i] if i < len(carriedProducts) else products

        sideReactants = removeOneListFromAnother(reactants, incomingMain)
        sideProducts = removeOneListFromAnother(products, outgoingMain)

        stepInfo.append(
            {
                "incomingMain": incomingMain,
                "outgoingMain": outgoingMain,
                "sideReactants": sideReactants,
                "sideProducts": sideProducts,
            }
        )
        mainChain.append(outgoingMain)

    return {"mainChain": mainChain, "stepInfo": stepInfo}


# =========================
# Drawing
# =========================
def drawChainedPathway(pathwayDict, outputPathPng=None, outputPathSvg=None):
    compressed = buildChainedPathway(pathwayDict)
    mainChain = compressed["mainChain"]
    stepInfo = compressed["stepInfo"]

    numNodes = len(mainChain)
    if numNodes == 0:
        return

    fig, ax = plt.subplots(figsize=(max(18, 5 * numNodes), 8))
    ax.axis("off")

    yMain = 4.0
    yTop = 6.1
    yBottom = 1.9

    xNodePositions = [2.5 + i * 5.2 for i in range(numNodes)]

    # Draw main chain nodes
    for nodeIdx, molSet in enumerate(mainChain):
        drawMolSet(ax, molSet, xNodePositions[nodeIdx], yMain, xStep=2.2, zoom=0.40)

    # Draw arrows and side species
    ruleList = pathwayDict["ruleList"]
    enthalpyList = pathwayDict["enthalpyList"]

    for stepIdx, step in enumerate(stepInfo):
        leftX = xNodePositions[stepIdx]
        rightX = xNodePositions[stepIdx + 1]

        arrowStart = leftX + 1.5
        arrowEnd = rightX - 0.8

        ax.annotate(
            "",
            xy=(arrowEnd, yMain),
            xytext=(arrowStart, yMain),
            arrowprops=dict(arrowstyle="->", lw=2.2)
        )

        ruleName = ruleList[stepIdx] if stepIdx < len(ruleList) else f"step_{stepIdx + 1}"
        enthalpy = enthalpyList[stepIdx] if stepIdx < len(enthalpyList) else ""

        stepLabel = f"Step {stepIdx + 1} | {ruleName}"
        if enthalpy:
            stepLabel += f" | {enthalpy}"

        ax.text(
            (arrowStart + arrowEnd) / 2,
            yMain + 0.85,
            stepLabel,
            ha="center",
            va="bottom",
            fontsize=10,
            fontweight="bold"
        )

        # Side reactants above arrow
        if len(step["sideReactants"]) > 0:
            drawMolSet(
                ax,
                step["sideReactants"],
                (arrowStart + arrowEnd) / 2 - 1.0,
                yTop,
                xStep=2.0,
                zoom=0.30
            )
            ax.plot(
                [(arrowStart + arrowEnd) / 2, (arrowStart + arrowEnd) / 2],
                [yTop - 0.6, yMain + 0.15],
                linestyle="--",
                linewidth=1.2
            )

        # Side products below arrow
        if len(step["sideProducts"]) > 0:
            drawMolSet(
                ax,
                step["sideProducts"],
                (arrowStart + arrowEnd) / 2 - 1.0,
                yBottom,
                xStep=2.0,
                zoom=0.30
            )
            ax.plot(
                [(arrowStart + arrowEnd) / 2, (arrowStart + arrowEnd) / 2],
                [yMain - 0.15, yBottom + 0.55],
                linestyle="--",
                linewidth=1.2
            )

    titleText = (
        f"Rank {pathwayDict['rank']} pathway"
        f" | final score = {pathwayDict['finalScore']}"
        f" | atomic economy = {pathwayDict['atomicEconomy']}"
    )

    ax.text(
        0.5, 1.03, titleText,
        transform=ax.transAxes,
        ha="center", va="bottom",
        fontsize=14, fontweight="bold"
    )

    ax.set_xlim(0, xNodePositions[-1] + 6.0)
    ax.set_ylim(0.5, 7.2)

    plt.tight_layout()

    if outputPathPng is not None:
        plt.savefig(outputPathPng, dpi=300, bbox_inches="tight")
    if outputPathSvg is not None:
        plt.savefig(outputPathSvg, bbox_inches="tight")

    plt.close(fig)


# =========================
# Driver
# =========================
def drawAllPathways(filePath):
    pathwayList = parsePathwayBlocks(filePath)
    print(f"Found {len(pathwayList)} pathways")

    for pathwayDict in pathwayList:
        rank = pathwayDict["rank"]

        pngPath = outputDir / f"pathway_rank_{rank}_chain.png"
        svgPath = outputDir / f"pathway_rank_{rank}_chain.svg"

        drawChainedPathway(
            pathwayDict,
            outputPathPng=pngPath,
            outputPathSvg=svgPath,
        )

        print(f"Saved PNG: {pngPath}")
        print(f"Saved SVG: {svgPath}")


if __name__ == "__main__":
    drawAllPathways(inputFile)

Found 4 pathways
Saved PNG: pathway_figures/pathway_rank_1_chain.png
Saved SVG: pathway_figures/pathway_rank_1_chain.svg
Saved PNG: pathway_figures/pathway_rank_2_chain.png
Saved SVG: pathway_figures/pathway_rank_2_chain.svg
Saved PNG: pathway_figures/pathway_rank_3_chain.png
Saved SVG: pathway_figures/pathway_rank_3_chain.svg
Saved PNG: pathway_figures/pathway_rank_4_chain.png
Saved SVG: pathway_figures/pathway_rank_4_chain.svg
